# Generate Thesis Device Geometries and nextnano++ Inputs

This generation-only notebook builds the seven thesis device layouts with the public qd_design pipeline, exposes their exact 2D polygons and polygon-preserving 3D structures, validates them, and can export geometry plus nextnano++ input artifacts.

Safety: nextnano is never executed here, and this notebook contains no simulation-result analysis. File writing is disabled by default and guarded by collision checks.

In [ ]:
import json
import re
import subprocess
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import display

from qd_design import (
    LinearDotArrayDevice,
    TopBarrierLinearDotArrayDevice,
    TwoDDotArrayDevice,
    build_simulation_layout,
    derive_quantum_region,
    make_reference_sige_ge_process_stack,
    plot_layout_spec_2d,
    plot_simulation_layout_3d,
    write_nextnano_input_from_template,
)

START_PATH = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in (START_PATH, *START_PATH.parents)
        if (candidate / "pyproject.toml").is_file()
        and (candidate / "src").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(
        f"Could not locate the repository root from {START_PATH}; "
        "expected a parent containing pyproject.toml and src/."
    )

GEOMETRY_OUTPUT_DIR = (
    REPO_ROOT / "data" / "gds" / "new_generated"
)
GENERATED_INPUT_DIR = (
    REPO_ROOT
    / "configs"
    / "robert_inputs"
    / "new_generated"
)
TEMPLATE_INPUT_PATH = (
    REPO_ROOT
    / "configs"
    / "robert_inputs"
    / "double_qd"
    / "3d"
    / "Double_Quantum_Dot_3D.in"
)

print("Repository root:", REPO_ROOT)
print("Template:", TEMPLATE_INPUT_PATH)

## Shared Geometry and Case Registry

CASES stores builder classes and plain, serializable keyword arguments. It deliberately stores no built device instances; the build cell creates a fresh builder for every selected case.

In [ ]:
device_y_size_nm = 200.0
ohmic_width_nm = 40.0
ohmic_length_nm = 200.0
barrier_width_nm = 40.0
barrier_length_nm = 140.0
plunger_body_width_nm = 40.0
plunger_body_length_nm = 50.0
plunger_head_top_width_nm = 60.0
plunger_head_max_width_nm = 100.0
plunger_head_height_nm = 100.0
plunger_upper_taper_height_nm = 25.0
plunger_lower_taper_height_nm = 25.0
barrier_to_plunger_gap_nm = 20.0

COMMON_GEOMETRY_KWARGS = {
    "device_y_size_nm": device_y_size_nm,
    "ohmic_width_nm": ohmic_width_nm,
    "ohmic_length_nm": ohmic_length_nm,
    "barrier_width_nm": barrier_width_nm,
    "barrier_length_nm": barrier_length_nm,
    "plunger_body_width_nm": plunger_body_width_nm,
    "plunger_body_length_nm": plunger_body_length_nm,
    "plunger_head_top_width_nm": plunger_head_top_width_nm,
    "plunger_head_max_width_nm": plunger_head_max_width_nm,
    "plunger_head_height_nm": plunger_head_height_nm,
    "plunger_upper_taper_height_nm": plunger_upper_taper_height_nm,
    "plunger_lower_taper_height_nm": plunger_lower_taper_height_nm,
    "barrier_to_plunger_gap_nm": barrier_to_plunger_gap_nm,
}

CASES = {
    "single_qd_bottom_barriers": {
        "builder_class": LinearDotArrayDevice,
        "builder_kwargs": {
            **COMMON_GEOMETRY_KWARGS,
            "name": "single_qd_bottom_barriers",
            "n_dots": 1,
            "ohmic_to_barrier_gap_nm": 20.0,
        },
        "expected_counts": {"plunger": 1, "barrier": 2, "ohmic": 2},
        "expected_domain_nm": (-170.0, 170.0, 0.0, 200.0),
        "nextnano_filename": "Single_Quantum_Dot_Bottom_Barriers_3D_from_PHIDL.in",
    },
    "lateral_dqd_bottom_barriers": {
        "builder_class": LinearDotArrayDevice,
        "builder_kwargs": {
            **COMMON_GEOMETRY_KWARGS,
            "name": "lateral_dqd_bottom_barriers",
            "n_dots": 2,
            "ohmic_to_barrier_gap_nm": 1000.0,
        },
        "expected_counts": {"plunger": 2, "barrier": 3, "ohmic": 2},
        "expected_domain_nm": (-1240.0, 1240.0, 0.0, 200.0),
        "nextnano_filename": "Lateral_Double_Quantum_Dot_Bottom_Barriers_3D_from_PHIDL.in",
    },
    "lateral_dqd_top_barriers": {
        "builder_class": TopBarrierLinearDotArrayDevice,
        "builder_kwargs": {
            **COMMON_GEOMETRY_KWARGS,
            "name": "lateral_dqd_top_barriers",
            "n_dots": 2,
            "ohmic_to_barrier_gap_nm": 1000.0,
        },
        "expected_counts": {"plunger": 2, "barrier": 3, "ohmic": 2},
        "expected_domain_nm": (-1240.0, 1240.0, 0.0, 200.0),
        "nextnano_filename": "Lateral_Double_Quantum_Dot_Top_Barriers_3D_from_PHIDL.in",
    },
    "qd_array_2x2": {
        "builder_class": TwoDDotArrayDevice,
        "builder_kwargs": {
            **COMMON_GEOMETRY_KWARGS,
            "name": "qd_array_2x2",
            "n_columns": 2,
            "row_gap_nm": 0.0,
            "ohmic_to_barrier_gap_nm": 20.0,
        },
        "expected_counts": {"plunger": 4, "barrier": 6, "ohmic": 4},
        "expected_domain_nm": (-260.0, 260.0, 0.0, 400.0),
        "nextnano_filename": "Quantum_Dot_Array_2x2_3D_from_PHIDL.in",
    },
    "qd_array_2x3": {
        "builder_class": TwoDDotArrayDevice,
        "builder_kwargs": {
            **COMMON_GEOMETRY_KWARGS,
            "name": "qd_array_2x3",
            "n_columns": 3,
            "row_gap_nm": 0.0,
            "ohmic_to_barrier_gap_nm": 20.0,
        },
        "expected_counts": {"plunger": 6, "barrier": 8, "ohmic": 4},
        "expected_domain_nm": (-350.0, 350.0, 0.0, 400.0),
        "nextnano_filename": "Quantum_Dot_Array_2x3_3D_from_PHIDL.in",
    },
    "single_qd_top_barriers": {
        "builder_class": TopBarrierLinearDotArrayDevice,
        "builder_kwargs": {
            **COMMON_GEOMETRY_KWARGS,
            "name": "single_qd_top_barriers",
            "n_dots": 1,
            "ohmic_to_barrier_gap_nm": 20.0,
        },
        "expected_counts": {"plunger": 1, "barrier": 2, "ohmic": 2},
        "expected_domain_nm": (-170.0, 170.0, 0.0, 200.0),
        "nextnano_filename": "Single_Quantum_Dot_Top_Barriers_3D_from_PHIDL.in",
    },
    "vertical_dqd_2x1": {
        "builder_class": TwoDDotArrayDevice,
        "builder_kwargs": {
            **COMMON_GEOMETRY_KWARGS,
            "name": "vertical_dqd_2x1",
            "n_columns": 1,
            "row_gap_nm": 0.0,
            "ohmic_to_barrier_gap_nm": 20.0,
        },
        "expected_counts": {"plunger": 2, "barrier": 4, "ohmic": 4},
        "expected_domain_nm": (-170.0, 170.0, 0.0, 400.0),
        "nextnano_filename": "Vertical_Double_Quantum_Dot_2x1_3D_from_PHIDL.in",
    },
}

## User Controls

Keep WRITE_FILES=False to inspect everything without writing. Change PREVIEW_CASE_KEY and rerun to inspect each selected case. STRUCTURE_Z_RANGE_NM changes only the 3D viewing window and never the simulated geometry; SHOW_FULL_STACK_3D=True adds a second rendering with z_range_nm=None so the complete substrate is visible.

The validated comparison run used a plunger voltage of -1.65 V. The default generated device inputs retain the established -3.0 V plunger value.

In [ ]:
# # TO BE REMOVED - ONLY ENABLE FOR A SINGLE CASE 

# # CUSTOM CASES: 1000 nm ohmic-to-barrier gap

# VERTICAL_DQD_1000NM_KEY = "vertical_dqd_2x1_1000nm_ohmic_gap"
# QD_ARRAY_2X2_1000NM_KEY = "qd_array_2x2_1000nm_ohmic_gap"
# QD_ARRAY_2X3_1000NM_KEY = "qd_array_2x3_1000nm_ohmic_gap"

# CUSTOM_CASE_KEYS = [
#     VERTICAL_DQD_1000NM_KEY,
#     QD_ARRAY_2X2_1000NM_KEY,
#     QD_ARRAY_2X3_1000NM_KEY,
# ]


# CASES[VERTICAL_DQD_1000NM_KEY] = {
#     "builder_class": TwoDDotArrayDevice,
#     "builder_kwargs": {
#         **CASES["vertical_dqd_2x1"]["builder_kwargs"],
#         "name": VERTICAL_DQD_1000NM_KEY,
#         "n_columns": 1,
#         "row_gap_nm": 0.0,
#         "ohmic_to_barrier_gap_nm": 1000.0,
#     },
#     "expected_counts": {
#         "plunger": 2,
#         "barrier": 4,
#         "ohmic": 4,
#     },
#     "expected_domain_nm": (-1150.0, 1150.0, 0.0, 400.0),
#     "nextnano_filename": (
#         "Vertical_Double_Quantum_Dot_2x1_"
#         "1000nm_Ohmic_Gap_3D_from_PHIDL.in"
#     ),
# }


# CASES[QD_ARRAY_2X2_1000NM_KEY] = {
#     "builder_class": TwoDDotArrayDevice,
#     "builder_kwargs": {
#         **CASES["qd_array_2x2"]["builder_kwargs"],
#         "name": QD_ARRAY_2X2_1000NM_KEY,
#         "n_columns": 2,
#         "row_gap_nm": 0.0,
#         "ohmic_to_barrier_gap_nm": 1000.0,
#     },
#     "expected_counts": {
#         "plunger": 4,
#         "barrier": 6,
#         "ohmic": 4,
#     },
#     "expected_domain_nm": (-1240.0, 1240.0, 0.0, 400.0),
#     "nextnano_filename": (
#         "Quantum_Dot_Array_2x2_"
#         "1000nm_Ohmic_Gap_3D_from_PHIDL.in"
#     ),
# }


# CASES[QD_ARRAY_2X3_1000NM_KEY] = {
#     "builder_class": TwoDDotArrayDevice,
#     "builder_kwargs": {
#         **CASES["qd_array_2x3"]["builder_kwargs"],
#         "name": QD_ARRAY_2X3_1000NM_KEY,
#         "n_columns": 3,
#         "row_gap_nm": 0.0,
#         "ohmic_to_barrier_gap_nm": 1000.0,
#     },
#     "expected_counts": {
#         "plunger": 6,
#         "barrier": 8,
#         "ohmic": 4,
#     },
#     "expected_domain_nm": (-1330.0, 1330.0, 0.0, 400.0),
#     "nextnano_filename": (
#         "Quantum_Dot_Array_2x3_"
#         "1000nm_Ohmic_Gap_3D_from_PHIDL.in"
#     ),
# }

In [ ]:
# # TO BE REMOVED - ONLY ENABLE FOR A SINGLE CASE

# # Generate only the three custom 1000 nm-gap cases.

# CASE_KEYS_TO_GENERATE = list(CUSTOM_CASE_KEYS)

# # This controls only which case is displayed in the 2D and 3D previews.
# # All three cases above will still be generated.
# PREVIEW_CASE_KEY = VERTICAL_DQD_1000NM_KEY

# WRITE_FILES = True
# OVERWRITE_EXISTING = False

# PLUNGER_VOLTAGE_V = -3.0
# OTHER_GATE_VOLTAGE_V = 0.0

# STRUCTURE_Z_RANGE_NM = (-170.0, 180.0)
# SHOW_FULL_STACK_3D = False

In [ ]:
CASE_KEYS_TO_GENERATE = list(CASES)
PREVIEW_CASE_KEY = "single_qd_bottom_barriers"

WRITE_FILES = False
OVERWRITE_EXISTING = False

PLUNGER_VOLTAGE_V = -3.0
OTHER_GATE_VOLTAGE_V = 0.0

STRUCTURE_Z_RANGE_NM = (-170.0, 180.0)
SHOW_FULL_STACK_3D = False

## Build Selected Cases and Summarize Them

Each selected registry entry is instantiated exactly once in this run from a fresh copy of its keyword arguments. Voltage overrides are derived from the resulting layout specification rather than from hard-coded label names.

In [ ]:
if not CASE_KEYS_TO_GENERATE:
    raise ValueError("Select at least one case in CASE_KEYS_TO_GENERATE.")
if len(CASE_KEYS_TO_GENERATE) != len(set(CASE_KEYS_TO_GENERATE)):
    raise ValueError("CASE_KEYS_TO_GENERATE must not contain duplicates.")
unknown_case_keys = [key for key in CASE_KEYS_TO_GENERATE if key not in CASES]
if unknown_case_keys:
    raise KeyError(f"Unknown case keys: {unknown_case_keys}")
if PREVIEW_CASE_KEY not in CASE_KEYS_TO_GENERATE:
    raise ValueError("PREVIEW_CASE_KEY must be one of the selected case keys.")

BUILT_CASES = {}
VOLTAGE_OVERRIDES_BY_CASE = {}
build_summary_rows = []

for case_key in CASE_KEYS_TO_GENERATE:
    case = CASES[case_key]
    builder = case["builder_class"](**dict(case["builder_kwargs"]))
    builder.ensure_built()
    layout_spec = builder.layout_spec()
    actual_counts = dict(Counter(element["gate_type"] for element in layout_spec))
    voltage_overrides = {
        element["voltage_label"]: (
            PLUNGER_VOLTAGE_V
            if element["gate_type"] == "plunger"
            else OTHER_GATE_VOLTAGE_V
        )
        for element in layout_spec
    }
    BUILT_CASES[case_key] = {
        "builder": builder,
        "layout_spec": layout_spec,
        "builder_summary": builder.summary(),
        "actual_counts": actual_counts,
    }
    VOLTAGE_OVERRIDES_BY_CASE[case_key] = voltage_overrides
    build_summary_rows.append(
        {
            "case_key": case_key,
            "builder": case["builder_class"].__name__,
            "plungers": actual_counts.get("plunger", 0),
            "barriers": actual_counts.get("barrier", 0),
            "ohmics": actual_counts.get("ohmic", 0),
            "elements": len(layout_spec),
            "polygons": sum(
                len(element["polygon_xy_nm"]) for element in layout_spec
            ),
        }
    )

BUILD_SUMMARY = pd.DataFrame(build_summary_rows)
display(BUILD_SUMMARY)

## Exact-Polygon 2D Preview

This interactive Plotly view draws the actual polygon vertices exported by the selected PHIDL builder. Change PREVIEW_CASE_KEY in the controls cell to inspect every case.

In [ ]:
LAYOUT_PREVIEW_FIGURE = plot_layout_spec_2d(
    BUILT_CASES[PREVIEW_CASE_KEY]["layout_spec"],
    title=f"{PREVIEW_CASE_KEY}: exact PHIDL polygons",
)
display(LAYOUT_PREVIEW_FIGURE)

## Reference Process Stack and Simulation Layouts

The shared reference stack supplies every vertical interface. Every selected 2D layout is converted to an actual SimulationLayout with zero lateral margins. The tables below report the background layers and every patterned region, including their complete z extents.

In [ ]:
PROCESS_STACK = make_reference_sige_ge_process_stack()
SIMULATION_LAYOUTS = {}
QUANTUM_REGIONS = {}

for case_key in CASE_KEYS_TO_GENERATE:
    simulation_layout = build_simulation_layout(
        name=f"{case_key}_simulation_layout",
        layout_elements=BUILT_CASES[case_key]["layout_spec"],
        process_stack=PROCESS_STACK,
        x_margin_nm=0.0,
        y_margin_nm=0.0,
    )
    SIMULATION_LAYOUTS[case_key] = simulation_layout
    QUANTUM_REGIONS[case_key] = derive_quantum_region(simulation_layout)

PROCESS_STACK_TABLE = pd.DataFrame(PROCESS_STACK.to_dict()["material_layers"])
BACKGROUND_REGION_TABLE = pd.DataFrame(
    [
        {
            "case_key": case_key,
            "name": region.name,
            "material": region.material,
            "z_min_nm": region.z_min_nm,
            "z_max_nm": region.z_max_nm,
            "contact_name": region.contact_name,
        }
        for case_key, simulation_layout in SIMULATION_LAYOUTS.items()
        for region in simulation_layout.background_regions
    ]
)
PATTERNED_REGION_TABLE = pd.DataFrame(
    [
        {
            "case_key": case_key,
            "name": region.name,
            "gate_type": region.gate_type,
            "layer_name": region.layer_name,
            "material": region.material,
            "voltage_label": region.voltage_label,
            "polygon_count": len(region.polygon_xy_nm),
            "vertex_counts": [
                len(polygon) for polygon in region.polygon_xy_nm
            ],
            "z_min_nm": region.z_min_nm,
            "z_max_nm": region.z_max_nm,
        }
        for case_key, simulation_layout in SIMULATION_LAYOUTS.items()
        for region in simulation_layout.patterned_regions
    ]
)

display(PROCESS_STACK_TABLE)
display(BACKGROUND_REGION_TABLE)
display(PATTERNED_REGION_TABLE)

## Polygon-Preserving 3D Structure Preview

The interactive renderer extrudes the exact 2D polygons at their process-stack z positions. STRUCTURE_Z_RANGE_NM only clips the view. When SHOW_FULL_STACK_3D is true, a second figure renders the complete substrate with z_range_nm=None.

In [ ]:
STRUCTURE_PREVIEW_FIGURE = plot_simulation_layout_3d(
    SIMULATION_LAYOUTS[PREVIEW_CASE_KEY],
    z_range_nm=STRUCTURE_Z_RANGE_NM,
    show_polygon_outlines=True,
)
display(STRUCTURE_PREVIEW_FIGURE)

if SHOW_FULL_STACK_3D:
    FULL_STACK_STRUCTURE_FIGURE = plot_simulation_layout_3d(
        SIMULATION_LAYOUTS[PREVIEW_CASE_KEY],
        z_range_nm=None,
        show_polygon_outlines=True,
    )
    display(FULL_STACK_STRUCTURE_FIGURE)

## Structural Preflight

All geometry, voltage, domain, process-stack, quantum-region, template, and path checks run before the guarded export cell. A failure stops the notebook before any artifact can be written.

In [ ]:
EXPECTED_BACKGROUND_Z_EXTENTS_NM = {
    "SiGe_body_contact": (-4115.0, -4015.0),
    "SiGe_buffer": (-4015.0, -15.0),
    "Ge_QW": (-15.0, 0.0),
    "SiGe_cap": (0.0, 101.0),
    "Al2O3_dielectric": (101.0, 173.0),
}
EXPECTED_PATTERNED_Z_EXTENTS_NM = {
    "barrier": (108.0, 138.0),
    "plunger": (143.0, 173.0),
    "ohmic": (-149.0, 173.0),
}

TARGET_PATHS_BY_CASE = {
    case_key: {
        "gds": GEOMETRY_OUTPUT_DIR / f"{case_key}.gds",
        "svg": GEOMETRY_OUTPUT_DIR / f"{case_key}.svg",
        "layout_spec": GEOMETRY_OUTPUT_DIR / f"{case_key}_layout_spec.json",
        "simulation_layout": (
            GEOMETRY_OUTPUT_DIR / f"{case_key}_simulation_layout.json"
        ),
        "generation_manifest": (
            GEOMETRY_OUTPUT_DIR / f"{case_key}_generation_manifest.json"
        ),
        "nextnano_input": (
            GENERATED_INPUT_DIR / CASES[case_key]["nextnano_filename"]
        ),
    }
    for case_key in CASE_KEYS_TO_GENERATE
}
RESOLVED_TARGET_PATHS_BY_CASE = {
    case_key: {name: path.resolve() for name, path in paths.items()}
    for case_key, paths in TARGET_PATHS_BY_CASE.items()
}
all_resolved_targets = [
    path
    for paths in RESOLVED_TARGET_PATHS_BY_CASE.values()
    for path in paths.values()
]
assert len(all_resolved_targets) == len(set(all_resolved_targets)), (
    "Generated target paths collide across cases."
)
all_input_filenames = [case["nextnano_filename"] for case in CASES.values()]
assert len(all_input_filenames) == len(set(all_input_filenames)), (
    "The case registry contains duplicate nextnano filenames."
)
assert TEMPLATE_INPUT_PATH.is_file(), f"Template not found: {TEMPLATE_INPUT_PATH}"

git_result = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
    check=False,
)
GIT_COMMIT = git_result.stdout.strip() if git_result.returncode == 0 else None

PREFLIGHT_ROWS = []
MANIFESTS_BY_CASE = {}
for case_key in CASE_KEYS_TO_GENERATE:
    case = CASES[case_key]
    record = BUILT_CASES[case_key]
    layout_spec = record["layout_spec"]
    simulation_layout = SIMULATION_LAYOUTS[case_key]
    domain = simulation_layout.domain
    quantum_region = QUANTUM_REGIONS[case_key]

    names = [element["name"] for element in layout_spec]
    voltage_labels = [element["voltage_label"] for element in layout_spec]
    assert len(names) == len(set(names)), f"{case_key}: duplicate element names"
    assert all(
        isinstance(label, str) and label.strip() for label in voltage_labels
    ), f"{case_key}: empty voltage label"
    assert len(voltage_labels) == len(set(voltage_labels)), (
        f"{case_key}: duplicate voltage labels"
    )
    assert record["actual_counts"] == case["expected_counts"], (
        f"{case_key}: expected {case['expected_counts']}, "
        f"got {record['actual_counts']}"
    )
    assert all(
        element["polygon_xy_nm"]
        and all(len(polygon) >= 3 for polygon in element["polygon_xy_nm"])
        for element in layout_spec
    ), f"{case_key}: nonempty polygon data required"

    actual_domain = (
        domain.x_min_nm,
        domain.x_max_nm,
        domain.y_min_nm,
        domain.y_max_nm,
    )
    assert actual_domain == case["expected_domain_nm"], (
        f"{case_key}: expected domain {case['expected_domain_nm']}, "
        f"got {actual_domain}"
    )
    assert domain.x_max_nm > domain.x_min_nm
    assert domain.y_max_nm > domain.y_min_nm
    assert domain.z_max_nm > domain.z_min_nm
    assert (
        domain.x_min_nm <= quantum_region.x_min_nm
        < quantum_region.x_max_nm <= domain.x_max_nm
        and domain.y_min_nm <= quantum_region.y_min_nm
        < quantum_region.y_max_nm <= domain.y_max_nm
        and domain.z_min_nm <= quantum_region.z_min_nm
        < quantum_region.z_max_nm <= domain.z_max_nm
    ), f"{case_key}: quantum region lies outside the simulation domain"

    background_by_name = {
        region.name: region for region in simulation_layout.background_regions
    }
    actual_background_extents = {
        name: (region.z_min_nm, region.z_max_nm)
        for name, region in background_by_name.items()
    }
    assert actual_background_extents == EXPECTED_BACKGROUND_Z_EXTENTS_NM
    assert background_by_name["SiGe_body_contact"].contact_name == "Body"
    assert (
        background_by_name["SiGe_body_contact"].z_max_nm
        == background_by_name["SiGe_buffer"].z_min_nm
        and background_by_name["SiGe_buffer"].z_max_nm
        == background_by_name["Ge_QW"].z_min_nm
        and background_by_name["Ge_QW"].z_max_nm
        == background_by_name["SiGe_cap"].z_min_nm
        and background_by_name["SiGe_cap"].z_max_nm
        == background_by_name["Al2O3_dielectric"].z_min_nm
    ), f"{case_key}: missing reference process-stack interface"
    assert all(
        (region.z_min_nm, region.z_max_nm)
        == EXPECTED_PATTERNED_Z_EXTENTS_NM[region.gate_type]
        for region in simulation_layout.patterned_regions
    ), f"{case_key}: unexpected patterned-region z extent"
    json.dumps(case["builder_kwargs"])

    generated_relative_paths = {
        name: path.relative_to(REPO_ROOT).as_posix()
        for name, path in TARGET_PATHS_BY_CASE[case_key].items()
    }
    MANIFESTS_BY_CASE[case_key] = {
        "case_key": case_key,
        "builder_class": case["builder_class"].__name__,
        "builder_parameters": dict(case["builder_kwargs"]),
        "builder_summary": record["builder_summary"],
        "voltage_overrides": VOLTAGE_OVERRIDES_BY_CASE[case_key],
        "template_path": TEMPLATE_INPUT_PATH.relative_to(REPO_ROOT).as_posix(),
        "git_commit": GIT_COMMIT,
        "simulation_domain": domain.to_dict(),
        "derived_quantum_region": quantum_region.to_dict(),
        "process_stack_description": PROCESS_STACK.to_dict(),
        "generated_relative_paths": generated_relative_paths,
    }
    PREFLIGHT_ROWS.append(
        {
            "case_key": case_key,
            "domain_nm": actual_domain,
            "quantum_region_nm": quantum_region.to_dict(),
            "targets": len(TARGET_PATHS_BY_CASE[case_key]),
            "status": "passed",
        }
    )

PREFLIGHT_SUMMARY = pd.DataFrame(PREFLIGHT_ROWS)
display(PREFLIGHT_SUMMARY)
print("All structural preflight checks passed before export.")

## Guarded Artifact Export

With WRITE_FILES=False this cell only lists the planned paths and performs no writes. With writing enabled, every target is resolved and checked first; any existing target aborts the complete export when OVERWRITE_EXISTING=False. Only after that all-clear are the two output directories created.

In [ ]:
WRITTEN_PATHS_BY_CASE = {}

if not WRITE_FILES:
    print("WRITE_FILES=False; no directories or files were created.")
    planned_rows = [
        {
            "case_key": case_key,
            "artifact": artifact_name,
            "path": path,
        }
        for case_key, paths in TARGET_PATHS_BY_CASE.items()
        for artifact_name, path in paths.items()
    ]
    display(pd.DataFrame(planned_rows))
else:
    existing_targets = sorted(
        path for path in all_resolved_targets if path.exists()
    )
    if existing_targets and not OVERWRITE_EXISTING:
        collision_text = "\n".join(f"- {path}" for path in existing_targets)
        raise FileExistsError(
            "Export aborted before writing because these targets exist:\n"
            f"{collision_text}"
        )

    GEOMETRY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    GENERATED_INPUT_DIR.mkdir(parents=True, exist_ok=True)

    for case_key in CASE_KEYS_TO_GENERATE:
        record = BUILT_CASES[case_key]
        paths = RESOLVED_TARGET_PATHS_BY_CASE[case_key]
        record["builder"].write_gds(str(paths["gds"]))
        record["builder"].write_svg(str(paths["svg"]))
        record["builder"].write_layout_spec_json(str(paths["layout_spec"]))
        SIMULATION_LAYOUTS[case_key].write_json(
            str(paths["simulation_layout"])
        )
        write_nextnano_input_from_template(
            simulation_layout=SIMULATION_LAYOUTS[case_key],
            template_path=TEMPLATE_INPUT_PATH,
            output_path=paths["nextnano_input"],
            voltage_overrides=VOLTAGE_OVERRIDES_BY_CASE[case_key],
        )
        paths["generation_manifest"].write_text(
            json.dumps(MANIFESTS_BY_CASE[case_key], indent=2) + "\n",
            encoding="utf-8",
        )
        WRITTEN_PATHS_BY_CASE[case_key] = dict(paths)

    print(f"Wrote {sum(len(paths) for paths in WRITTEN_PATHS_BY_CASE.values())} artifacts.")

## Validate Written Inputs and Summarize Generation

When files were written, this cell checks the generated nextnano++ text against the layout polygons, labels, contacts, derived quantum bounds, and the template solver controls. In dry-run mode it reports successful preflight without reading or creating artifacts.

In [ ]:
GENERATED_INPUT_VALIDATION = {}
validation_failures = []

if WRITE_FILES:
    template_text = TEMPLATE_INPUT_PATH.read_text(encoding="utf-8")
    solver_names = ("strain", "poisson", "quantum", "quantum_poisson")
    template_solver_matches = {
        name: re.search(
            r"(?m)^\$" + name + r"\s*=\s*([^#\n]+)", template_text
        )
        for name in solver_names
    }

    quantum_bounds_pattern = re.compile(
        r'name\s*=\s*"c-Ge_QW"\s*'
        r'x\s*=\s*\[([-+0-9.eE]+),\s*([-+0-9.eE]+)\]\s*'
        r'y\s*=\s*\[([-+0-9.eE]+),\s*([-+0-9.eE]+)\]\s*'
        r'z\s*=\s*\[([-+0-9.eE]+),\s*([-+0-9.eE]+)\]',
        flags=re.DOTALL,
    )

    for case_key in CASE_KEYS_TO_GENERATE:
        input_path = RESOLVED_TARGET_PATHS_BY_CASE[case_key]["nextnano_input"]
        text = input_path.read_text(encoding="utf-8")
        layout_spec = BUILT_CASES[case_key]["layout_spec"]
        simulation_layout = SIMULATION_LAYOUTS[case_key]
        labels = [element["voltage_label"] for element in layout_spec]
        expected_polygon_count = sum(
            len(region.polygon_xy_nm)
            for region in simulation_layout.patterned_regions
        )
        generated_solver_matches = {
            name: re.search(
                r"(?m)^\$" + name + r"\s*=\s*([^#\n]+)", text
            )
            for name in solver_names
        }
        solver_controls_present = all(
            template_solver_matches[name] is not None
            and generated_solver_matches[name] is not None
            for name in solver_names
        )
        solver_controls_preserved = solver_controls_present and all(
            template_solver_matches[name].group(1).strip()
            == generated_solver_matches[name].group(1).strip()
            for name in solver_names
        )

        quantum_match = quantum_bounds_pattern.search(text)
        parsed_quantum_bounds = None
        if quantum_match is not None:
            values = [float(value) for value in quantum_match.groups()]
            parsed_quantum_bounds = {
                "x_min_nm": values[0],
                "x_max_nm": values[1],
                "y_min_nm": values[2],
                "y_max_nm": values[3],
                "z_min_nm": values[4],
                "z_max_nm": values[5],
            }

        checks = {
            "nonempty_file": input_path.is_file() and bool(text.strip()),
            "all_voltage_labels_defined": all(
                re.search(
                    r"(?m)^\$" + re.escape(label) + r"\s*=", text
                )
                for label in labels
            ),
            "all_patterned_regions_represented": all(
                f"# patterned region: {region.name}" in text
                for region in simulation_layout.patterned_regions
            ),
            "polygonal_prism_count_matches": (
                text.count("polygonal_prism{") == expected_polygon_count
            ),
            "body_contact_present": "contact{ name = Body }" in text,
            "remove_surface_charge_present": (
                "# auxiliary fermi_hole contact: remove_surface_charge" in text
                and "contact{ name = remove_surface_charge }" in text
            ),
            "zero_fermi_QW_present": (
                "# auxiliary fermi_hole contact: zero_fermi_QW" in text
                and "contact{ name = zero_fermi_QW }" in text
            ),
            "quantum_bounds_match": (
                parsed_quantum_bounds == QUANTUM_REGIONS[case_key].to_dict()
            ),
            "solver_controls_preserved": solver_controls_preserved,
            "generated_run_block_present": (
                "run{" in text
                and all("!WHEN $" + name in text for name in solver_names)
            ),
            "all_case_targets_written": all(
                path.is_file()
                for path in RESOLVED_TARGET_PATHS_BY_CASE[case_key].values()
            ),
            "no_unresolved_path_collision": (
                len(all_resolved_targets) == len(set(all_resolved_targets))
            ),
        }
        failed_checks = [name for name, passed in checks.items() if not passed]
        status = "passed" if not failed_checks else f"failed: {failed_checks}"
        GENERATED_INPUT_VALIDATION[case_key] = {
            "checks": checks,
            "status": status,
        }
        if failed_checks:
            validation_failures.append((case_key, failed_checks))
else:
    GENERATED_INPUT_VALIDATION = {
        case_key: {
            "checks": {},
            "status": "preflight passed; dry run, no files written",
        }
        for case_key in CASE_KEYS_TO_GENERATE
    }

FINAL_GENERATION_SUMMARY = pd.DataFrame(
    [
        {
            "case_key": case_key,
            "builder": CASES[case_key]["builder_class"].__name__,
            "plungers": BUILT_CASES[case_key]["actual_counts"].get("plunger", 0),
            "barriers": BUILT_CASES[case_key]["actual_counts"].get("barrier", 0),
            "ohmics": BUILT_CASES[case_key]["actual_counts"].get("ohmic", 0),
            "domain_nm": SIMULATION_LAYOUTS[case_key].domain.to_dict(),
            "quantum_region_nm": QUANTUM_REGIONS[case_key].to_dict(),
            "output_filename": CASES[case_key]["nextnano_filename"],
            "validation_status": GENERATED_INPUT_VALIDATION[case_key]["status"],
        }
        for case_key in CASE_KEYS_TO_GENERATE
    ]
)
display(FINAL_GENERATION_SUMMARY)

if validation_failures:
    raise AssertionError(f"Generated-input validation failures: {validation_failures}")